# LLM Class: Customer Support Copilot

## Instructor-led notebook: privacy-aware triage, RAG, and tool use

This class focuses only on the project's LLM-style support tasks: safe ticket triage, retrieval-augmented generation (RAG) over policy documents, and a tool-using refund assistant. It is separate from the machine-learning class in [machine_learning_concepts.ipynb](./machine_learning_concepts.ipynb), which covers RFM segmentation and lapse prediction.

The project uses the deterministic local `mock_llm.py`; it does not call an external model or require an API key. Data and policies are synthetic. Refund actions only write local demonstration records and do not move money.

### Before class

1. Use Python 3.11 or newer and install dependencies with `python -m pip install -r requirements.txt`.
2. Open and run this notebook from the repository root.
3. Run cells in order. Part D resets the local demonstration ledger when run.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    candidates = [ROOT / "instamart-customer-copilot", ROOT.parent]
    ROOT = next((candidate for candidate in candidates if (candidate / "generate_data.py").exists()), ROOT)
if not (ROOT / "generate_data.py").exists():
    raise FileNotFoundError("Open this notebook from the repository root.")

print("Project root:", ROOT)
print("Top-level files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

def run_script(relative_path):
    """Run a repository script with the current notebook interpreter."""
    subprocess.run([sys.executable, str(ROOT / relative_path)], cwd=ROOT, check=True)

## 1. Prepare the local demo data

Ticket triage reads the generated ticket export. The refund-agent demo reads the cleaned orders and SQLite database, so this setup generates the synthetic inputs and runs the project's cleaning and validation stages first.

In [8]:
import pandas as pd

run_script("generate_data.py")
run_script("src/part_a_clean.py")
run_script("src/part_a_validate.py")

## 2. LLM task: privacy-first support-ticket triage

### Concepts and safeguards

- **Classification:** the assistant assigns each ticket a bounded category and status; unsupported or invalid results should not be treated as trustworthy.
- **Data minimization:** email addresses, phone numbers, and order IDs are redacted before text is sent to the local classifier. Pattern-based redaction is useful but cannot guarantee removal of every sensitive detail.
- **Safe handling:** blank and overlong tickets are stopped before classification, transient failures are retried, categories are validated, and review-required cases are routed to people. Fraud routing takes precedence over ordinary customer-segment routing.
- **Operational constraints:** a token budget is tracked, and fallback behavior is intentionally limited rather than inventing certainty.

The demonstration uses a local deterministic mock classifier; it is not an external LLM integration.

In [3]:
run_script("src/part_c_triage.py")
triage_dir = ROOT / "outputs"
triage_budget = pd.read_csv(triage_dir / "triage_budget_demo.csv")
triage_zero_shot = pd.read_csv(triage_dir / "triage_zero_shot_eval.csv")
triage_results = pd.read_csv(triage_dir / "triage_results.csv")
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))
print("Budget-demo rows:", len(triage_budget), "Zero-shot rows:", len(triage_zero_shot))

,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At-Risk Low Spend,HIGH,security_team,81
1,T0073,C0080,classified_json,billing,Frequent Shoppers,NORMAL,support_queue,77
2,T0095,C1087,classified_json,technical,Occasional Shoppers,NORMAL,support_queue,79
3,T0159,C1172,skipped_empty,NaN,Frequent Shoppers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional Shoppers,HIGH,security_team,81
5,T0124,C0139,classified_json,fraud,Frequent Shoppers,HIGH,security_team,81
6,T0061,C0812,classified_json,billing,Frequent Shoppers,NORMAL,support_queue,73
7,T0118,C0355,classified_json,delivery,Frequent Shoppers,NORMAL,support_queue,76
8,T0037,C1156,classified_json,billing,Premium Spenders,HIGH,premium_desk,77
9,T0002,C0881,classified_json,delivery,Occasional Shoppers,NORMAL,support_queue,75


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,flagged_long,4
4,skipped_empty,4


Budget-demo rows: 164 Zero-shot rows: 164


### Discussion: privacy and safe routing

- Why should invalid categories go to human review instead of being guessed?
- Why should fraud routing take precedence over Premium-customer routing?
- Why block empty or overlong messages before a classifier call?
- What sensitive information could remain even after pattern-based redaction?

## 3. LLM task: policy RAG and a refund agent

### Concepts and safeguards

- **Retrieval-augmented generation (RAG):** policy text is split into overlapping chunks, then TF-IDF and cosine similarity retrieve relevant context for the local policy assistant. Answers should be grounded in retrieved policy rather than unsupported guesses.
- **TF-IDF:** term frequency (TF) measures how often a term appears in a chunk, while inverse document frequency (IDF) gives more weight to terms that occur in fewer chunks and discounts common terms. A limitation is lexical matching: equivalent phrasings such as “late” and “delayed” may use different words and retrieve poorly.
- **Tool use:** `TOOLS_SPEC` declares each tool's name, description, and JSON parameter schema; `TOOL_MAP` dispatches the model-requested name to its Python function. The tools look up orders, retrieve policies, calculate a refund, issue a simulated refund, or escalate to a person.
- **Safe stopping and resumability:** eligibility rules, refund limits, and model-call, token, and elapsed-time limits constrain the workflow. The agent checkpoints the full conversation after each tool observation so a resumed run can continue without repeating completed actions.

All documents, orders, and agent responses in this exercise are synthetic. The refund ledger is a local teaching artifact, not a payment system.

In [4]:
run_script("src/part_d_agent.py")

### Inspect policy retrieval and evaluation

The policy evaluation compares retrieved source documents and expected answer facts for the questions in `data/raw/policy_eval.csv`. Discuss why chunk overlap can help facts near boundaries and why lexical TF-IDF retrieval can miss synonyms or paraphrases.

In [5]:
policy_eval = pd.read_csv(ROOT / "outputs" / "policy_eval_results.csv")
display(policy_eval.head(10))
answerable = policy_eval[policy_eval.expected_doc.ne("NONE")]
unanswerable = policy_eval[policy_eval.expected_doc.eq("NONE")]
print("Expected-document hits:", int(answerable.doc_hit.sum()), "/", len(answerable))
print("Expected-fact hits:", int(answerable.fact_hit.sum()), "/", len(answerable))
print("Unanswerable questions:", len(unanswerable))
display(unanswerable[["question", "answer"]])

,question,expected_doc,expected_fact,retrieved_docs,answer,doc_hit,fact_hit
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes,refund_policy.txt|refund_policy.txt|privacy_po...,"order is delivered in more than 45 minutes, th...",True,True
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead,refund_policy.txt|refund_policy.txt|refund_pol...,A refund above INR 1000 must be approved by a ...,True,True
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days,refund_policy.txt|privacy_policy.txt|refund_po...,A refund request must be raised within 7 days ...,True,True
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25,delivery_policy.txt|delivery_policy.txt|paymen...,A delivery fee of INR 25 is charged on orders ...,True,True
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more,delivery_policy.txt|delivery_policy.txt|paymen...,"of INR 25 is charged on orders below INR 199, ...",True,True
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password,delivery_policy.txt|delivery_policy.txt|paymen...,Electronics items are delivered only after the...,True,True
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500,payment_policy.txt|payment_policy.txt|payment_...,Cash on delivery is available only for orders ...,True,True
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours,payment_policy.txt|payment_policy.txt|refund_p...,"If a payment fails but money is deducted, the ...",True,True
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash,payment_policy.txt|payment_policy.txt|payment_...,Wallet balance can be used for any order and i...,True,True
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days,privacy_policy.txt|privacy_policy.txt|delivery...,Delivery addresses are deleted from active sys...,True,True


Expected-document hits: 12 / 12
Expected-fact hits: 12 / 12
Unanswerable questions: 2


,question,answer
12,What is the favourite colour of the company ma...,I could not find that information in the polic...
13,Who won the cricket world cup final?,I could not find that information in the polic...


### Inspect refund tools, scenarios, and safe stopping

The tool loop maps tool names to Python functions: order lookup, policy search, refund calculation, refund issue, and human escalation. Only completed orders delivered **after 45 minutes** qualify for a full refund; refunds above INR 1,000 are escalated. Tool executions are checkpointed so a resumed run does not repeat completed actions. Token, time, and iteration limits provide safe handoffs.

The ledger is a local JSONL demonstration file; no money moves. The scenario script also demonstrates a simulated crash/resume and timeout/token-limit fallbacks.

In [6]:
agent_runs_path = ROOT / "outputs" / "agent_runs.jsonl"
agent_runs = pd.read_json(agent_runs_path, lines=True)
display(agent_runs[["scenario_id", "status", "final_answer", "tools"]])
ledger_path = ROOT / "outputs" / "refund_ledger.jsonl"
print("Local refund-ledger entries:", sum(1 for _ in ledger_path.open(encoding="utf-8")) if ledger_path.exists() else 0)
print("Checkpoint files:", sorted(p.name for p in (ROOT / "state").glob("*.json")))

,scenario_id,status,final_answer,tools
0,S1,REFUND,REFUND: INR 128.25 issued.,"get_order,search_policy,calculate_refund,issue..."
1,S2,REFUND,REFUND: INR 135.0 issued.,"get_order,search_policy,calculate_refund,issue..."
2,S3,ESCALATE,ESCALATE: refund above INR 1000 sent to a huma...,"get_order,search_policy,calculate_refund,escal..."
3,S4,DENIED,DENIED: delivery was within 45 minutes.,"get_order,search_policy,calculate_refund"
4,S5,DENIED,DENIED: only completed orders can be refunded.,"get_order,search_policy"
5,S6,NOT_FOUND,NOT_FOUND: no such order.,get_order
6,S7,FALLBACK_MAX_ITERATIONS,FALLBACK_MAX_ITERATIONS: hand off to a human a...,"search_policy,search_policy,search_policy,sear..."


Local refund-ledger entries: 4
Checkpoint files: ['S1.json', 'S2.json', 'S3.json', 'S4.json', 'S5.json', 'S6.json', 'S7.json', 'demo_crash_resume.json', 'demo_timeout.json', 'demo_token_limit.json']
